# Text-to-SQL: Traducir Lenguaje Natural a Consultas SQL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexfazio/InteligenciaArtificial/blob/main/tutoriales/notebooks/text-to-sql/01-text-to-sql-basico.ipynb)

Construye un sistema que traduce preguntas en español a SQL, las ejecuta y devuelve respuestas en lenguaje natural usando Claude y SQLite.

In [ ]:
# Instalar dependencias
%pip install anthropic sqlalchemy pandas duckdb --quiet

In [ ]:
import os
import re
import sqlite3
import pandas as pd
import anthropic

# Configurar API key
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # O usar variable de entorno
client = anthropic.Anthropic()
print('Cliente Anthropic inicializado.')

In [ ]:
# Crear base de datos de demostración en memoria
conn = sqlite3.connect(':memory:')

conn.executescript("""
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY, nombre TEXT NOT NULL,
        email TEXT, ciudad TEXT, fecha_registro TEXT
    );
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY, nombre TEXT NOT NULL,
        categoria TEXT, precio REAL, stock INTEGER
    );
    CREATE TABLE pedidos (
        id INTEGER PRIMARY KEY, cliente_id INTEGER,
        fecha TEXT, total REAL, estado TEXT,
        FOREIGN KEY (cliente_id) REFERENCES clientes(id)
    );
    INSERT INTO clientes VALUES
        (1,'Ana García','ana@mail.com','Madrid','2024-01-15'),
        (2,'Luis Martínez','luis@mail.com','Barcelona','2024-02-20'),
        (3,'María López','maria@mail.com','Valencia','2024-03-10'),
        (4,'Carlos Ruiz','carlos@mail.com','Madrid','2024-04-05');
    INSERT INTO productos VALUES
        (1,'Laptop Pro','Electrónica',1299.99,15),
        (2,'Monitor 4K','Electrónica',449.99,30),
        (3,'Teclado Mecánico','Periféricos',89.99,50),
        (4,'Mouse Inalámbrico','Periféricos',39.99,75),
        (5,'Auriculares BT','Audio',129.99,20);
    INSERT INTO pedidos VALUES
        (1,1,'2024-05-01',1299.99,'completado'),
        (2,2,'2024-05-03',539.98,'completado'),
        (3,1,'2024-05-10',89.99,'completado'),
        (4,3,'2024-05-15',169.98,'pendiente'),
        (5,4,'2024-05-20',449.99,'enviado');
""")
conn.commit()
print('Base de datos creada con 3 tablas y datos de demo.')

In [ ]:
def extraer_esquema_compacto(conn):
    """Extrae el esquema en formato compacto para incluir en el prompt."""
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tablas = [row[0] for row in cursor.fetchall()]
    
    partes = []
    for tabla in tablas:
        cursor.execute(f"PRAGMA table_info({tabla})")
        columnas = cursor.fetchall()
        cols = ', '.join(f"{c[1]} {c[2]}" for c in columnas)
        partes.append(f"{tabla}({cols})")
    return '\n'.join(partes)

esquema = extraer_esquema_compacto(conn)
print('Esquema compacto:')
print(esquema)

In [ ]:
# Validación de SQL antes de ejecutar
OPERACIONES_PELIGROSAS = [
    r'\bDROP\b', r'\bDELETE\b', r'\bUPDATE\b', r'\bINSERT\b',
    r'\bTRUNCATE\b', r'\bALTER\b', r'\bCREATE\b', r'--', r'/\*'
]

def validar_sql(sql):
    """Devuelve (True, '') si el SQL es seguro, (False, motivo) si no."""
    sql_upper = sql.upper()
    for patron in OPERACIONES_PELIGROSAS:
        if re.search(patron, sql_upper):
            op = patron.replace(r'\b', '')
            return False, f'Operación bloqueada: {op}'
    sql_limpio = sql.strip().lstrip('(')
    if not (sql_limpio.upper().startswith('SELECT') or sql_limpio.upper().startswith('WITH')):
        return False, 'Solo se permiten consultas SELECT o WITH'
    return True, ''

# Probar la validación
casos = [
    ('SELECT * FROM clientes', True),
    ('DROP TABLE clientes', False),
    ('UPDATE productos SET precio=0', False),
]
for sql, esperado in casos:
    valido, razon = validar_sql(sql)
    ok = '✓' if valido == esperado else '✗'
    print(f'{ok} {"VÁLIDO" if valido else "BLOQUEADO"}: {sql[:45]}')
    if razon:
        print(f'   → {razon}')

In [ ]:
FEW_SHOT = """
Ejemplos:
P: ¿Cuántos clientes hay en total?
SQL: SELECT COUNT(*) AS total_clientes FROM clientes;

P: ¿Cuáles son los 3 productos más caros?
SQL: SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 3;

P: ¿Cuál es el total de ventas completadas?
SQL: SELECT SUM(total) AS ventas_completadas FROM pedidos WHERE estado = 'completado';
"""

def generar_sql(pregunta, esquema):
    respuesta = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        system=f"""Eres un experto en SQL. Convierte preguntas a SQL válido para SQLite.

Esquema:
{esquema}
{FEW_SHOT}
Responde SOLO con la consulta SQL, sin markdown ni explicaciones.""",
        messages=[{'role': 'user', 'content': pregunta}]
    )
    sql = respuesta.content[0].text.strip()
    if sql.startswith('```'):
        sql = '\n'.join(sql.split('\n')[1:-1]).strip()
    return sql

def interpretar_resultado(pregunta, sql, df):
    resultado_str = df.to_string(index=False) if not df.empty else '(sin resultados)'
    respuesta = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        messages=[{
            'role': 'user',
            'content': f'Pregunta: {pregunta}\nResultado:\n{resultado_str}\n\nResponde en una oración clara en español.'
        }]
    )
    return respuesta.content[0].text.strip()

print('Funciones definidas.')

In [ ]:
def query(pregunta, conn, esquema):
    """Pipeline completo: NL → SQL → ejecutar → NL."""
    resultado = {'sql': None, 'dataframe': None, 'respuesta': None, 'error': None}
    try:
        sql = generar_sql(pregunta, esquema)
        resultado['sql'] = sql
        
        valido, razon = validar_sql(sql)
        if not valido:
            resultado['error'] = f'SQL rechazado: {razon}'
            return resultado
        
        df = pd.read_sql_query(sql, conn)
        resultado['dataframe'] = df
        resultado['respuesta'] = interpretar_resultado(pregunta, sql, df)
    except Exception as e:
        resultado['error'] = str(e)
    return resultado

# Prueba
preguntas = [
    '¿Cuántos clientes hay en total?',
    '¿Cuáles son los 3 productos más caros?',
    '¿Cuántos pedidos hay por estado?',
    '¿Qué ciudad tiene más clientes?',
]

for pregunta in preguntas:
    print(f'\nPregunta: {pregunta}')
    res = query(pregunta, conn, esquema)
    if res['error']:
        print(f'Error: {res["error"]}')
    else:
        print(f'SQL: {res["sql"]}')
        print(f'Respuesta: {res["respuesta"]}')

In [ ]:
# Limitaciones del sistema básico
print('=== Limitaciones conocidas ===')
print()
print('1. JOINs complejos (4+ tablas): precisión ~60%')
print('   Solución: schema-linking (Artículo 02)')
print()
print('2. Fechas relativas ("el mes pasado"): no funciona sin CURRENT_DATE')
print('   Solución: inyectar fecha actual en el prompt (Artículo 02)')
print()
print('3. Agregaciones anidadas: requieren few-shot específicos')
print()
print('4. Dialecto SQL: este sistema usa SQLite. Para PostgreSQL/MySQL,')
print('   especificar el dialecto en el system prompt.')

## Resumen

Construiste un pipeline Text-to-SQL básico con:
- **Inyección de esquema** compacta para ahorrar tokens
- **Validación** que bloquea operaciones destructivas (DROP, DELETE, UPDATE)
- **Few-shot examples** para mejorar la precisión
- **Interpretación** del resultado en lenguaje natural

**Siguiente**: `02-text-to-sql-avanzado.ipynb` — Self-correction y esquemas complejos